# Geographic Analysis

This notebook examines the geographic distribution of CRM deals and explores whether German-language level is associated with deal success.

### Business questions
- Which cities generate the largest deal volumes?
- How concentrated is demand across German cities?
- Does observed conversion differ by German-language level?
- Are city-level language patterns stable enough to support business decisions?

> **Caution:** language level is not randomly assigned and may be missing or recorded selectively. The results show association, not causation.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from helpers import colors

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

pd.options.display.max_columns = None


## Load Processed Data

In [ ]:
deals = pd.read_pickle(PROCESSED_DIR / 'deals_clean.pkl')
buyers = pd.read_pickle(PROCESSED_DIR / 'buyers.pkl')

## 1. Deal Distribution by City

In [ ]:
# Keep records with a known city
known_cities = deals[(deals['city'] != 'Unknown')].copy()

# Calculate city-level deal volume and win rate
city_stats = (known_cities.groupby('city').agg(total=('id', 'count'), won=('stage_group', lambda x: (x == 'Won').sum()),).reset_index())
city_stats['win_rate'] = (city_stats['won'] / city_stats['total'] * 100).round(1)
city_stats = city_stats[city_stats['total'] >= 5].sort_values('total', ascending=False)

print(f"Cities with ≥5 deals: {len(city_stats)}")
city_stats.head(10)

In [ ]:
plt.figure(figsize=(10, 5))
top_cities = city_stats.head(10)

sns.scatterplot(data=top_cities, x='total', y='win_rate', size='total', sizes=(100, 1000), color=colors['accent'], alpha=0.7)

for x, y, city in zip(
    top_cities['total'],
    top_cities['win_rate'],
    top_cities['city']
):
    plt.text(x, y, city, fontsize=8)

plt.title('City Performance')
plt.xlabel('Total Deals')
plt.ylabel('Win Rate (%)')

plt.tight_layout()
plt.legend().remove()
plt.show()

In [ ]:
top10_share = city_stats.head(10)['total'].sum() / known_cities.shape[0] * 100
print(f'Top 10 cities account for {top10_share:.1f}% of deals with a known city')

### Geographic Insights

- The top 10 cities account for a substantial share of deals with a known city, showing clear demand concentration in major urban areas.
- **Berlin** has the largest observed deal volume.
- Other high-volume cities include major German cities such as München, Hamburg, Nürnberg, Leipzig, Düsseldorf, Dresden, Frankfurt, Dortmund, and Köln.
- The dataset also contains a long tail of cities with only a few recorded deals.

## 2. German-Language Level

In [ ]:
# Count potential deals by city and language level
total = (deals[['city', 'level_of_deutsch']].value_counts().reset_index(name='total_deals'))

In [ ]:
# Count confirmed buyers by city and language level
won = (buyers[['city', 'level_of_deutsch']].value_counts().reset_index(name='won_deals'))

In [ ]:
conversion_by_city_language = total.merge(won,on=['city', 'level_of_deutsch'],how='left')

conversion_by_city_language['won_deals'] = (conversion_by_city_language['won_deals'].fillna(0))

conversion_by_city_language['conversion_%'] = (conversion_by_city_language['won_deals'] / conversion_by_city_language['total_deals']* 100).round(1)

In [ ]:
conversion_by_city_language

In [ ]:
df = conversion_by_city_language[(conversion_by_city_language['city'] != 'Unknown') &(conversion_by_city_language['level_of_deutsch'] != 'Unknown')]

In [ ]:
df_filtered = df[df['total_deals'] >= 10]

In [ ]:
top_conversion = (df_filtered.sort_values(by='conversion_%', ascending=False))
top_conversion.head(20)

### City × Language Insights

Among city-language combinations with sufficient observations, B1 frequently appears among successful deals and in several cities shows higher observed conversion than A2.

However:
- sample sizes vary substantially by city,
- language data may be missing or selectively recorded,
- and language level can correlate with other customer characteristics.

Therefore, these results should be interpreted as exploratory associations rather than evidence that language level itself causes higher conversion.

To validate the pattern at a broader level, the next step compares conversion by language level without splitting by city.

In [ ]:
# Potential deals by language level
total_lang = (deals['level_of_deutsch'].value_counts().reset_index())

total_lang.columns = ['level_of_deutsch', 'total_deals']
total_lang

In [ ]:
# Confirmed buyers by language level
won_lang = (buyers['level_of_deutsch'].value_counts().reset_index())
won_lang.columns = ['level_of_deutsch', 'won_deals']
won_lang

In [ ]:
language_conversion = total_lang.merge(won_lang,on='level_of_deutsch', how='left')
language_conversion

In [ ]:
language_conversion['conversion_%'] = (language_conversion['won_deals'] / language_conversion['total_deals']* 100).round(1)

In [ ]:
language_conversion = language_conversion[language_conversion['level_of_deutsch'] != 'Unknown']

In [ ]:
language_conversion.sort_values(by='conversion_%',ascending=False)

### Overall Language-Level Insights

Within records where German-language level is known, observed conversion differs across levels. B1 shows a higher conversion rate than A2 in the analyzed data, while higher levels do not show a consistent monotonic increase.

Results for levels with small sample sizes should be interpreted cautiously. The analysis does **not** establish a causal effect of German proficiency on purchase behavior.